# LUMEN-299 · 5 — What the programme decided

**The failure.** Every new measurement of the speed of light disagreed with the last one.
Read one at a time at the conventional five per cent, a century of sixteen experiments
declares a disagreement about once a decade, and most of those declarations are noise
dressed as findings. Read as a programme, two of them survive — and they are the two that
mattered.

This notebook is the programme's view: every run filed with its conformance, the one
artifact that crossed from one laboratory's scope to another's, the check that refuses
to pool an in-air number with in-vacuo ones, and the decision stream corrected as a
stream.

| | reaches into |
|---|---|
| the century, filed | `build.ExperimentBuilder`, `io.ExperimentRun`, `io.Catalog.find` |
| crossing a scope | `io.Catalog.transfer`, `io.Catalog.crossings`, `io.CrossScopeError` |
| what each record is a record of | `meta.commensurable`, `io.DefinitionRegistry` |
| the disagreements, one at a time | `design.Readout`, `design.program_decisions`, `design.from_p_value`, `design.online_decisions` |
| what would have been reported | `calibrate.combine_inverse_variance`, `display.Card` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import pandas as pd

import lightspeed as L

import tempfile
from pathlib import Path

from scipy import stats as _st

from axiom.build import ExperimentBuilder, readouts_across
from axiom.calibrate import Measurement, combine_inverse_variance
from axiom.core import LedgerLine, TimeWindow
from axiom.design import Readout, from_p_value, online_decisions, program_decisions
from axiom.io import Catalog, CrossScopeError, DefinitionRegistry, Program
from axiom.meta import Corpus, commensurable, record_from_summary

from axiom.display import Card, enable, show, table

sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, CRITICAL, ORANGE, caption, compare, figure, intervals, mark_x, mark_y

enable();  # every axiom result renders itself from here on

world = L.simulate()
catalog = Catalog(Path(tempfile.mkdtemp()) / "lumen")
programs = L.programs()
builders = {}
for lab, program in programs.items():
    catalog.register(program)
    builders[lab] = ExperimentBuilder(catalog, program)
print(f"{len(builders)} laboratories in {len(L.FAMILIES)} families")

## 1 · The century, filed

Every experiment goes through the same four moves — plan, commit, start, read — in its
own laboratory's scope, with the estimand it actually measured and the season it ran.
Two runs carry the stories notebook 3 told: Mount Wilson's re-survey is a filed deviation,
and the vacuum tube's change is not filed at all.

In [ ]:
runs = {}
for r in world.readouts:
    e = r.experiment
    b = builders[e.lab]
    est = L.estimand(medium=e.medium, radiation=e.radiation)
    roles = {"season": TimeWindow(start=e.year, stop=e.year + 1, basis="per_period")}
    if e.baseline_km is not None:
        roles["survey"] = L.BaselineSurvey(length_km=e.baseline_km, relative_error=1e-5, surveyed_by=e.who, year=e.year)
    run = b.start(b.commit(b.plan(e.key, estimand=est, roles=roles, at=f"{e.year}-01-01T00:00:00+00:00"),
                           at=f"{e.year}-02-01T00:00:00+00:00"), at=f"{e.year}-03-01T00:00:00+00:00")
    if e.key == "michelson_1926":
        run = run.deviate("survey", L.MOUNT_WILSON_RESURVEY, reason="re-surveyed in August", state="asserted",
                          at="1926-08-10T00:00:00+00:00")
    if e.key == "pease_pearson_1935":
        run = run.model_copy(update={"roles": {**run.roles, "survey": L.MOUNT_WILSON.content_hash()}})
    measurement = Measurement(estimand=est, estimate=r.read, se=r.se, source=e.key, method=L.FAMILY_LABEL[e.family])
    runs[e.key] = b.read(run, measurement, at=f"{e.year}-12-01T00:00:00+00:00")

table([[run.experiment, run.scope, run.stage, run.conformance().status] for run in runs.values()],
      headers=("experiment", "scope", "stage", "conformance"), title="sixteen runs, as filed")
licensed = readouts_across(list(builders.values()))
print(f"{len(licensed)} of {len(runs)} runs are licensed evidence; left out:",
      sorted(set(runs) - {r.experiment for r in licensed}))

## 2 · Crossing a scope

The 1935 vacuum tube was built by Michelson's own laboratory, and the only baseline
survey good enough to plan it against was the 1926 one. An artifact reaching a second
scope is a transfer, the ledger line is not optional, and `crossings()` is where somebody
finds it six months later. A Kerr-cell laboratory asking for the same artifact gets a
`CrossScopeError` naming both scopes — not `None`, and not the survey.

In [ ]:
michelson, pease = programs["michelson"], programs["pease-pearson"]
survey_digest = catalog.store(michelson).put(L.MOUNT_WILSON_RESURVEY, label="survey:1926", tags={"role": "survey"})
line = LedgerLine(
    kind="scope_crossing",
    statement="the 1926 Coast and Geodetic Survey baseline is reused to plan the 1935 vacuum tube",
    detail={"agreement": "same laboratory, same instrument family"},
)
catalog.transfer(survey_digest, source=michelson, target=pease, line=line)
print("held by:", catalog.scope_of(survey_digest))
table([[e.scope, e.label, e.derived_from[:12] + "…"] for e in catalog.crossings()],
      headers=("into scope", "label", "licensed by"), title="every artifact that crossed a boundary")
try:
    catalog.store(programs["anderson"]).get(survey_digest)
except CrossScopeError as err:
    print("\nCrossScopeError:", err)

## 3 · What each record is a record of

Notebook 4 corrected every in-air number to vacuo before it entered the corpus. Suppose it
had not — which is what happens when the correction lives in a footnote. Two checks catch
it, and they catch different amounts.

The estimand check reads each record's `Estimand` against the reference and finds the
medium as a `version` difference: `downgraded`, under an assumption it cannot evaluate.
The definition check goes one level back, to what each laboratory *registered* the term
to mean, and finds two definitions in the corpus: `blocked`.

In [ ]:
definitions = DefinitionRegistry(catalog)
by_lab = {}
for r in world.readouts:
    e = r.experiment
    est = L.estimand(medium=e.medium, radiation=e.radiation)
    definitions.register(e.program, "c", est, at=f"{e.year}-01-01")
    by_lab[e.key] = (e.lab, est)

uncorrected = Corpus(name="as published", records=tuple(
    record_from_summary(study=r.experiment.key, contributor=r.experiment.lab, quantity="relative_change",
                        estimate=L.ppm(r.read), se=r.se / L.C_TRUE * 1e6, read="experiment",
                        family="speed-of-light", period=str(r.experiment.year), source=r.experiment.who)
    for r in world.readouts if r.experiment.year < 1950))
estimands = {key: est for key, (lab, est) in by_lab.items()}
lab_programs = {lab: programs[lab] for lab in {v[0] for v in by_lab.values()}}

by_estimand = commensurable(uncorrected, estimands, reference="michelson_1926")
print("estimand check :", by_estimand.verdict().status, "| assumptions:", [a.name for a in by_estimand.assumptions])
by_definition = commensurable(uncorrected, estimands, reference="michelson_1926",
                              definitions=definitions, programs=lab_programs, term="c")
print("definition check:", by_definition.verdict().status)
table([[i.study, i.facet, i.status, i.reason[:80]] for i in by_definition.entries],
      headers=("record", "facet", "status", "why the pool refuses"))

Four records, all before 1903, all 88 km/s low by definition. The estimand check would
have let them through under `version_irrelevance`, and the pool of notebook 4's section 3
would have been shifted by every one of them. The registry knows what the type cannot:
those laboratories *said* what they were measuring, and it was not the same thing.

## 4 · The disagreements, one at a time

Each experiment arrives and is compared with the inverse-variance consensus of everything
licensed before it. The $z$ of that comparison is a p-value, and a p-value is a decision
the programme is about to take: *does this experiment overturn the consensus?* Sixteen
such decisions at five per cent is the programme being wrong most of the time it says
yes, and `program_decisions` puts the uncorrected count beside the corrected one.

In [ ]:
ordered = [r for r in world.readouts]
consensus_of = {}
readouts = []
rows = []
prior_values, prior_ses = [], []
for r in ordered:
    e = r.experiment
    licensed_here = runs[e.key].conformance().licensed
    if prior_values:
        c_mean, c_se = combine_inverse_variance(prior_values, prior_ses)
        zscore = (r.read_vacuo - c_mean) / math.sqrt(r.se**2 + c_se**2)
        p = float(2 * _st.norm.sf(abs(zscore)))
        readouts.append(Readout(experiment=e.key, party=e.family, p_value=p, evalue=from_p_value(p)))
        rows.append([e.year, e.who, f"{c_mean:,.1f} ± {c_se:.1f}", f"{r.read_vacuo:,.1f} ± {r.se:.1f}",
                     f"{zscore:+.1f}", f"{p:.3g}", "" if licensed_here else "not licensed"])
    if licensed_here:
        prior_values.append(r.read_vacuo)
        prior_ses.append(r.se)
    consensus_of[e.year] = combine_inverse_variance(prior_values, prior_ses)
table(rows, headers=("year", "arrival", "consensus before it", "what it found", "z", "p", ""),
      title="each experiment against everything licensed before it")

In [ ]:
reports = {m: program_decisions(readouts, alpha=0.05, method=m, period="1849-1972") for m in ("none", "holm", "e_bh")}
table([[m, ", ".join(d.experiment for d in rep.decisions if d.go) or "—", len(rep.go),
        f"{rep.expected_false_go_uncorrected:.2f}"] for m, rep in reports.items()],
      headers=("method", "declared a real disagreement", "count", "expected false at 5%"),
      title="fifteen decisions, three ways of taking them")
stream = online_decisions(readouts, alpha=0.05, stream="the century, in arrival order")
print("LORD, deciding each on arrival:", stream.rejected)
show(reports["e_bh"].ledger_line())

Uncorrected, the programme declares three disagreements real. Holm keeps two, and so does
LORD deciding each on arrival; e-BH on e-values calibrated from p-values keeps one, which
is the calibrator's price (note 0035) for needing nothing about how the readouts depend on
each other. The two that survive the p-value routes are the two that history remembers:
Mount Wilson against the Annapolis generation, and Bergstrand's geodimeter against the
bandwagon. Anderson's 1937 Kerr cell, the third uncorrected finding, is one careful
experiment disagreeing with the systematic of the last. The programme did not need to know
the answer to sort them; it needed to count its own decisions.

## 5 · What would have been reported

The consensus as the programme would have stated it in 1941 and in 1972, with the number
it would have printed and the number it was entitled to.

In [ ]:
def report(year):
    licensed_before = [(r.read_vacuo, r.se) for r in world.readouts
                       if r.experiment.year <= year and runs[r.experiment.key].conformance().licensed]
    mean, se = combine_inverse_variance([v for v, _ in licensed_before], [s for _, s in licensed_before])
    return mean, se, len(licensed_before)

rows = []
for year in (1882, 1926, 1941, 1951, 1972):
    mean, se, n = report(year)
    rows.append([year, n, f"{mean:,.2f} ± {se:.2f}", f"{(L.C_TRUE - mean) / se:+.1f}"])
table(rows, headers=("as of", "licensed records", "inverse-variance consensus (km/s)", "truth, in se"),
      title="what the programme would have printed")

m41, s41, n41 = report(1941)
m72, s72, n72 = report(1972)
card = Card(title="LUMEN-299 — the speed of light, as a programme", status="good")
card.add("in 1941", f"{m41:,.1f} ± {s41:.1f} km/s from {n41} licensed records — {(L.C_TRUE - m41) / s41:+.1f} se from the truth", emphasis=True)
card.add("in 1972", f"{m72:,.4f} ± {s72:.4f} km/s from {n72} records", emphasis=True)
card.add("the truth", f"{L.C_TRUE:,.3f} km/s, defined 1983")
card.add("disagreements that survived correction", ", ".join(reports["e_bh"].go))
card.add("runs excluded as unlicensed", ", ".join(sorted(set(runs) - {r.experiment for r in licensed})))
card.add("records refused by definition", str(len(by_definition.entries)))
card.note = ("The 1941 number is precise, agreed, and low. Nothing in the pooling could have "
             "said so; the programme's only defence was to value an instrument from outside "
             "every family in the book, and to count its own decisions.")
show(card)

## What the case study says

**The question.** One constant, five kinds of instrument, a century: how far should the
next experiment believe the ones before it, and how does a programme tell a finding from a
fluctuation when every instrument carries a systematic its own error bar cannot see?

**What the quantity is.** Distance covered per second of transit, a `ratio`, with the
medium as the intervention's version and the radiation as the population — and a unit
registered with its conversion, because Fizeau's third figure is a league.

**Whose it is.** Party = instrument family, programme = laboratory; every run planned,
committed, started and read in its own scope; one artifact crossed under a ledger line;
one departure filed and one not, and only one of those runs is evidence.

**What a reading is worth.** A property of the baseline and its survey, not of patience;
Mount Wilson bought its nats with a mountain, and readings could not have settled 1879
against 1882 because the disagreement was not in the readings.

**What the pool sees.** A systematic shared within a family is a party-level variance
component and `three_level` finds it — when the families disagree. When they share it,
nothing in the hierarchy can, and the honest number for the next experimenter is the
prediction interval, or the `new_party` prior, which is the one the bandwagon got wrong.

**What the programme decided.** Fifteen disagreements at five per cent, three declared;
two survive Holm and LORD, one survives e-BH, and the two are Mount Wilson against the
Annapolis generation and Bergstrand against the bandwagon. The 1941 consensus was
299,776 ± 4 in history; in this world the five bandwagon records pool to the same number
and the programme's whole book to 299,786 ± 3, both low, the second saved from being
wrong by four standard errors only by the weight of 1926. The 1972 one was right to a
part in a billion. The difference between them is not a better pool. It is a new kind of
instrument, filed where it could be found.